In [9]:
!pip -q install transformers accelerate

In [10]:
import torch
from transformers import  AutoTokenizer , AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = "auto"
).to(device)

model.eval()

print("LLM Agent model loaded!")
print("Device:", device)




config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM Agent model loaded!
Device: cpu


In [11]:
def calculator(a,b):
  return a * b


def knowledge_search(topic):

  knowledge = {
        "python": "Python is a popular programming language.",
        "machine learning": "Machine learning allows computers to learn from data.",
        "deep learning": "Deep learning uses neural networks.",
        "rag": "RAG retrieves external information to help an LLM answer."
  }

  return knowledge.get(
        topic.lower(),
        "Information not found."
    )




In [12]:
import json

def choose_tool(user_question):

    messages = [
        {
            "role": "system",
            "content": """
You are an AI agent that selects tools.

Available tools:

1. calculator
   Use when the user asks to multiply two numbers.
   Arguments: a, b

2. knowledge_search
   Use when the user asks about Python, machine learning,
   deep learning, or RAG.
   Argument: topic

Return ONLY valid JSON.

Calculator example:
{"tool": "calculator", "arguments": {"a": 25, "b": 4}}

Search example:
{"tool": "knowledge_search", "arguments": {"topic": "python"}}
"""
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    print("LLM decision:", response)

    return json.loads(response)

In [13]:
def execute_tool(tool_request):

    tool_name = tool_request["tool"]
    arguments = tool_request["arguments"]

    if tool_name == "calculator":

        a = arguments["a"]
        b = arguments["b"]

        if not isinstance(a, (int, float)):
            raise ValueError("Invalid first number")

        if not isinstance(b, (int, float)):
            raise ValueError("Invalid second number")

        return calculator(a, b)

    elif tool_name == "knowledge_search":

        topic = arguments["topic"]

        allowed_topics = [
            "python",
            "machine learning",
            "deep learning",
            "rag"
        ]

        if topic.lower() not in allowed_topics:
            return "Topic not supported."

        return knowledge_search(topic)

    else:
        raise ValueError("Unknown tool requested!")

In [14]:
def run_agent(question):

    print("User:", question)

    tool_request = choose_tool(question)

    result = execute_tool(tool_request)

    print("\nTool result:", result)

    return result

In [15]:
run_agent("Please multiply 25 and 4")

User: Please multiply 25 and 4
LLM decision: {"tool": "calculator", "arguments": {"a": 25, "b": 4}}

Tool result: 100


100